In [ ]:
import os
import soundfile as sf
import pandas as pd
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torchvision.models import resnet18

In [ ]:
submission_path = r'/kaggle/input/competitions/birdclef-2026/test_soundscapes'
test_files = sorted(os.listdir(submission_path))

num_segments = 12
sr = 32_000
segment_len = 5 * sr
device = torch.device('cuda')


to_db = T.AmplitudeToDB()
to_mel_low = T.MelSpectrogram(
    sample_rate=sr,
    n_fft=2048,
    hop_length=512,
    n_mels=128,
    f_min=0,
    f_max=10_000
)
to_mel_high = T.MelSpectrogram(
    sample_rate=sr,
    n_fft=2048,
    hop_length=512,
    n_mels=128,
    f_min=8_000,
    f_max=sr // 2
)

def get_sps(y):

    with torch.no_grad():
        mel_low = to_mel_low(y)
        mel_high = to_mel_high(y)

    stacked = torch.stack([
        scaler(mel_low),
        scaler(mel_high)
    ], dim=0)
    return stacked.float()

def scaler(mel: torch.Tensor):
    log_mel = to_db(mel)
    min_ = log_mel.min()
    max_ = log_mel.max()
    if (max_ - min_) == 0: return torch.zeros_like(log_mel)
    return (log_mel - min_) / (max_ - min_)

In [ ]:
model = resnet18()

old_conv = model.conv1
model.conv1 = nn.Conv2d(2, 64, 7, 2, 3, bias=False)

with torch.no_grad():
    model.conv1.weight[:] = old_conv.weight[:, :2]

model.fc = nn.Linear(512, 234)

state = torch.load('best_label.pth', map_location=device)
model.load_state_dict(state, strict=False)
model.to(device)
model.eval()

In [ ]:
with open('/kaggle/input/birdclef-2026/sample_submission.csv') as f:
    clses = f.readline().strip().split(',')[1:]
    
rows = []
for file in test_files:

    file_path = os.path.join(submission_path, file)

    y, _ = sf.read(file_path, dtype="float32")
    y = torch.from_numpy(y)

    file_id = file.replace(".ogg", "")

    for start in range(num_segments):

        shift = start * segment_len
        segment = y[shift:shift+segment_len]

        if len(segment) == 0:
            continue

        time = (start + 1) * 5
        row_id = f"{file_id}_{time}"

        sps = get_sps(segment).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(sps)
            probs = torch.sigmoid(logits).cpu().numpy()[0]

        rows.append([row_id] + probs.tolist())

In [ ]:
df = pd.DataFrame(rows, columns=['row_id'] + clses)
df.to_csv('submission.csv', index=False)